# ragfallback — 10 Silent RAG Failures, Caught Live

[![PyPI](https://img.shields.io/pypi/v/ragfallback)](https://pypi.org/project/ragfallback/)
[![GitHub](https://img.shields.io/badge/GitHub-ragfallback-black?logo=github)](https://github.com/irfanalidv/ragfallback)

This notebook runs every major module in **ragfallback v2.0** on real public data (SQuAD Wikipedia, CC BY-SA 4.0).  
No paid API keys needed. Just **Runtime → Run all** and watch each failure mode get caught.

| Cell | What it covers |
|------|----------------|
| 1 | Install |
| 2 | Load real data (SQuAD Wikipedia) |
| 3 | Ingest pipeline — ChunkQualityChecker + EmbeddingGuard + sanitize_documents |
| 4 | Build index + RetrievalHealthCheck |
| 5 | SmartThresholdHybridRetriever + FailoverRetriever |
| 6 | ContextWindowGuard + OverlappingContextStitcher |
| 7 | StaleIndexDetector |
| 8 | RAGEvaluator — recall, faithfulness, overall score |
| 9 | Full pipeline summary |

In [ ]:
# Cell 1 — Install
# Takes ~60s on Colab. Warnings about dependency conflicts are safe to ignore.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ragfallback[chroma,huggingface,real-data,hybrid]"],
    check=True
)
print("\n✅ Install complete")

In [ ]:
# Cell 2 — Suppress noisy warnings then load 50 real Wikipedia passages from SQuAD
import warnings, logging, os
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset
from langchain_core.documents import Document

print("Loading SQuAD Wikipedia passages...")
ds = load_dataset("rajpurkar/squad", split="validation", trust_remote_code=True)

seen, docs, probes = set(), [], []
for row in ds:
    ctx = row["context"].strip()
    if ctx not in seen and len(seen) < 50:
        seen.add(ctx)
        docs.append(Document(page_content=ctx, metadata={"source": "squad"}))
    if row["answers"]["text"] and len(probes) < 20:
        probes.append({
            "question": row["question"],
            "ground_truth": row["answers"]["text"][0]
        })

print(f"✅ Loaded {len(docs)} real Wikipedia passages")
print(f"✅ Loaded {len(probes)} real Q&A pairs")
print(f"\nSample passage (first 200 chars):")
print(f"  {docs[0].page_content[:200]}...")
print(f"\nSample Q&A:")
print(f"  Q: {probes[0]['question']}")
print(f"  A: {probes[0]['ground_truth']}")

In [ ]:
# Cell 3 — Ingest pipeline
# Catches: bad chunks, dimension mismatches, corrupt metadata — before anything hits the index
from ragfallback.diagnostics import (
    ChunkQualityChecker,
    EmbeddingGuard,
    sanitize_documents,
)
from langchain_huggingface import HuggingFaceEmbeddings

print("=" * 55)
print("INGEST PIPELINE")
print("=" * 55)

# 3a — Chunk quality
print("\n[1/3] ChunkQualityChecker — scanning 50 passages...")
report = ChunkQualityChecker(min_chars=100, max_chars=8000).check(docs)
print(report.summary())
if report.has_issues:
    fixed_docs = ChunkQualityChecker().auto_fix(docs)
    print(f"  Auto-fixed → {len(fixed_docs)} clean chunks")
else:
    fixed_docs = docs
    print("  ✅ All chunks passed — safe to embed")

# 3b — Embedding guard
print("\n[2/3] EmbeddingGuard — validating dimensions before index write...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
result = EmbeddingGuard(expected_dim=384).validate(embeddings)
if result.passed:
    print("  ✅ Dimension 384 confirmed — no index corruption risk")
else:
    print(f"  ❌ Guard failed: {result.error}")

# 3c — Metadata sanitizer
print("\n[3/3] sanitize_documents — normalising metadata for vector store write...")
# Inject a dirty doc to demonstrate
dirty = Document(
    page_content="Test passage with dirty metadata.",
    metadata={"tags": ["rag", "test"], "scores": {"relevance": 0.9}, "source": "demo"}
)
clean = sanitize_documents([dirty])
print(f"  Before: {dirty.metadata}")
print(f"  After:  {clean[0].metadata}")
print("  ✅ Metadata is now JSON-safe for Chroma / Pinecone / Qdrant")

clean_docs = sanitize_documents(fixed_docs)
print(f"\n✅ Ingest pipeline complete — {len(clean_docs)} passages ready for indexing")

In [ ]:
# Cell 4 — Build index + RetrievalHealthCheck
# Verifies the index actually returns correct answers before you serve traffic
from langchain_community.vectorstores import Chroma
from ragfallback.diagnostics import RetrievalHealthCheck

print("=" * 55)
print("INDEX BUILD + RETRIEVAL HEALTH CHECK")
print("=" * 55)

print("\nBuilding Chroma index on 50 Wikipedia passages...")
store = Chroma.from_documents(clean_docs, embeddings)
print("✅ Index built")

print("\nRunning RetrievalHealthCheck on 10 real Q&A probes...")
probe_dict = {p["question"]: p["ground_truth"][:60] for p in probes[:10]}
health = RetrievalHealthCheck(k=4).run_substring_probes(store, probe_dict)

print(f"\n  Hit rate   : {health.hit_rate:.0%}")
print(f"  Avg latency: {health.avg_latency_ms:.0f} ms per query")

if health.hit_rate >= 0.8:
    print("\n✅ Index is healthy — safe to serve")
else:
    print("\n⚠️  Low hit rate — check embeddings or chunk quality before serving")

In [ ]:
# Cell 5 — SmartThresholdHybridRetriever + FailoverRetriever
# SmartThreshold: falls back to BM25 when dense scores are weak
# Failover: switches to secondary retriever when primary fails or returns empty
from ragfallback.retrieval import SmartThresholdHybridRetriever, FailoverRetriever

print("=" * 55)
print("HYBRID RETRIEVAL + FAILOVER")
print("=" * 55)

# SmartThreshold hybrid
print("\n[1/2] SmartThresholdHybridRetriever")
print("  Dense threshold: 0.3 | BM25 fallback enabled")
hybrid = SmartThresholdHybridRetriever.from_documents(
    clean_docs, embeddings, dense_threshold=0.3, k=4
)
q = probes[0]["question"]
results = hybrid.invoke(q)
print(f"  Query : {q}")
print(f"  Chunks returned: {len(results)}")
print(f"  ✅ Retrieval succeeded (BM25 fallback available if dense scores drop)")

# Failover
print("\n[2/2] FailoverRetriever — simulating primary outage...")

class BrokenRetriever:
    """Simulates a primary retriever that always fails."""
    def invoke(self, query, **kwargs):
        raise ConnectionError("Primary vector store unreachable")

fallback_retriever = store.as_retriever(search_kwargs={"k": 4})
failover = FailoverRetriever(
    primary=BrokenRetriever(),
    fallback=fallback_retriever,
    min_results=1
)
results = failover.invoke(q)
print(f"  Primary raised ConnectionError")
print(f"  Failover triggered → secondary returned {len(results)} chunks")
print(f"  ✅ No empty response served to user")

In [ ]:
# Cell 6 — ContextWindowGuard + OverlappingContextStitcher
# Guard: trims retrieved chunks to fit the LLM token budget
# Stitcher: merges adjacent chunks so cross-boundary answers aren't split
from ragfallback.diagnostics import ContextWindowGuard, OverlappingContextStitcher

print("=" * 55)
print("CONTEXT WINDOW GUARD + CONTEXT STITCHER")
print("=" * 55)

retrieved = store.as_retriever(search_kwargs={"k": 6}).invoke(probes[1]["question"])

# ContextWindowGuard
print("\n[1/2] ContextWindowGuard (gpt-4o preset — 128k token budget)")
guard = ContextWindowGuard.from_model_name("gpt-4o")
selected, report = guard.select(probes[1]["question"], retrieved, embeddings)
print(f"  Retrieved : {len(retrieved)} chunks")
print(f"  After trim: {len(selected)} chunks fit within token budget")
print(f"  ✅ No context overflow — LLM won't hallucinate missing content")

# OverlappingContextStitcher
print("\n[2/2] OverlappingContextStitcher")
# Create two adjacent chunks from same source to demonstrate stitching
passage = clean_docs[0].page_content
mid = len(passage) // 2
chunk_a = Document(page_content=passage[:mid + 50], metadata={"source": "squad", "chunk_id": 0})
chunk_b = Document(page_content=passage[mid - 50:], metadata={"source": "squad", "chunk_id": 1})
merged = OverlappingContextStitcher().stitch([chunk_a, chunk_b])
print(f"  Input : 2 adjacent chunks from same source (overlap at boundary)")
print(f"  Output: {len(merged)} merged chunk")
print(f"  ✅ Cross-boundary answer preserved — no answer lost at chunk split")

In [ ]:
# Cell 7 — StaleIndexDetector
# Catches when source files changed since last index build (SHA256 manifest)
import tempfile, pathlib
from ragfallback.diagnostics import StaleIndexDetector

print("=" * 55)
print("STALE INDEX DETECTOR")
print("=" * 55)

with tempfile.TemporaryDirectory() as tmpdir:
    # Create a fake source doc
    doc_path = pathlib.Path(tmpdir) / "policy.md"
    doc_path.write_text("Refund policy: 30 days full refund.")
    manifest_path = pathlib.Path(tmpdir) / "manifest.json"

    det = StaleIndexDetector(manifest_path=str(manifest_path))

    # Record at index build time
    det.record_paths([str(doc_path)])
    print("\n  Index built — manifest recorded")

    # Check — no changes yet
    report = det.check_paths([str(doc_path)])
    print(f"  Check 1 (no changes): stale={report.has_stale} ✅")

    # Simulate document update
    doc_path.write_text("Refund policy UPDATED: 14 days refund only.")
    report = det.check_paths([str(doc_path)])
    print(f"  Check 2 (file updated): stale={report.has_stale} ⚠️")
    print(f"  {report.summary()}")
    print("  ✅ Stale index caught — rebuild triggered before serving stale answers")

In [ ]:
# Cell 8 — RAGEvaluator
# Scores recall@k, faithfulness, and overall quality — no LLM judge, no external APIs
from ragfallback.evaluation import RAGEvaluator

print("=" * 55)
print("RAG EVALUATOR")
print("=" * 55)

retriever = store.as_retriever(search_kwargs={"k": 4})
ev = RAGEvaluator()
scores = []

print("\nEvaluating on 10 real SQuAD Q&A pairs...\n")
for probe in probes[:10]:
    question = probe["question"]
    ground_truth = probe["ground_truth"]
    retrieved_docs = retriever.invoke(question)
    answer = retrieved_docs[0].page_content if retrieved_docs else "Not found"
    score = ev.evaluate(
        question, answer,
        [d.page_content for d in retrieved_docs],
        ground_truth=ground_truth,
    )
    scores.append(score)

print(ev.batch_summary(scores))
print("✅ Evaluation complete — no LLM judge, no API key, no cost")

In [ ]:
# Cell 9 — Full pipeline summary
print("=" * 55)
print("RAGFALLBACK v2.0 — PIPELINE SUMMARY")
print("=" * 55)

modules = [
    ("ChunkQualityChecker",            "Ingest",     "Blocks bad chunks before embedding"),
    ("EmbeddingGuard",                 "Ingest",     "Catches dimension mismatches before index write"),
    ("sanitize_documents",             "Ingest",     "JSON-safe metadata for any vector store"),
    ("RetrievalHealthCheck",           "Index",      "Verifies hit rate before serving traffic"),
    ("StaleIndexDetector",             "Index",      "SHA256 manifest — catches stale indexes"),
    ("SmartThresholdHybridRetriever",  "Retrieval",  "BM25 fallback when dense scores are weak"),
    ("FailoverRetriever",              "Retrieval",  "Secondary retriever on primary outage"),
    ("ContextWindowGuard",             "Generation", "Trims chunks to fit LLM token budget"),
    ("OverlappingContextStitcher",     "Generation", "Merges adjacent chunks — no answer lost at boundary"),
    ("RAGEvaluator",                   "Evaluation", "recall@k, faithfulness, nDCG — no external APIs"),
]

print(f"\n{'Module':<38} {'Stage':<12} {'What it prevents'}")
print("-" * 90)
for name, stage, desc in modules:
    print(f"  {name:<36} {stage:<12} {desc}")

print("\n" + "=" * 55)
print("All 10 failure modes demonstrated on real data ✅")
print("=" * 55)
print("\npip install ragfallback[chroma,huggingface,real-data]")
print("https://github.com/irfanalidv/ragfallback")